# anomaly_scan · Varredura de variáveis em janelas de disparo

Compara o comportamento de todos os PI points da planta em dois contextos:

- **Baseline normal** — mês anterior completo (calculado dinamicamente)
- **Janelas de anomalia** — N dias antes de cada disparo registrado no SharePoint (`Gatilhos_Selagem`)

**Saída:** `anomaly_scan.csv` com z-score por sinal × janela e ranking dos sinais mais anômalos.

**Filtros:** PI points com `Capsule` no nome são ignorados. Sinais sem cobertura mínima no baseline são descartados.

**Pré-requisitos:** sessão `spy` autenticada no Seeq Data Lab.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from seeq import spy

# ── Parâmetros ────────────────────────────────────────────────────────────
JANELA_ANTES_D  = 7      # dias antes do disparo incluídos na janela
JANELA_DEPOIS_D = 0      # dias após o disparo
GRID            = '1h'   # granularidade do pull
PILOT_GRID      = '1d'   # granularidade do check de disponibilidade
MIN_COBERTURA   = 0.10   # cobertura mínima para sinal ser considerado ativo
ZSCORE_LIMIAR   = 2.0    # |z| acima do qual o sinal é classificado como anômalo
TRIGGER_DESDE   = None   # None = início do ano vigente (calculado abaixo)

SEARCH_PATTERN   = '*BRSZA020*'
EXCLUIR_CAPSULES = True
OUTPUT_CSV       = 'anomaly_scan.csv'

# ── Timezone e janela temporal ─────────────────────────────────────────────
user_tz = spy.session.get_user_timezone()
_agora  = pd.Timestamp.now(tz=user_tz)

ANO_VIGENTE = _agora.year

# Pull contínuo: 1 jan do ano vigente → agora
DATA_INICIO = pd.Timestamp(f'{ANO_VIGENTE}-01-01', tz=user_tz)
DATA_FIM    = _agora

# Baseline: mês anterior completo (fatiado localmente do pull contínuo)
_ini_mes_atual = _agora.replace(day=1)
_fim_baseline  = _ini_mes_atual - pd.Timedelta(days=1)
_ini_baseline  = _ini_mes_atual - pd.DateOffset(months=1)
DATA_BASELINE_START = _ini_baseline
DATA_BASELINE_END   = _fim_baseline

# TRIGGER_DESDE: início do ano vigente por padrão
if TRIGGER_DESDE is None:
    TRIGGER_DESDE = DATA_INICIO

print(f'Timezone          : {user_tz}')
print(f'Pull contínuo     : {DATA_INICIO.date()} → {DATA_FIM.date()}')
print(f'Baseline (fatia)  : {DATA_BASELINE_START.date()} → {DATA_BASELINE_END.date()}')
print(f'Janela disparo    : -{JANELA_ANTES_D}d / +{JANELA_DEPOIS_D}d')
print(f'Disparos desde    : {pd.Timestamp(TRIGGER_DESDE).date()}')
print(f'Busca             : {SEARCH_PATTERN}')
print(f'Cobertura min     : {MIN_COBERTURA:.0%}')

## 1. Datas de disparo

Ordem de tentativa:
1. SharePoint `Gatilhos_Selagem` — fonte de produção real (requer `sharepoint.ev`)
2. `anomaly_delay.csv` — backtest local (coluna `is_yellow_card`)
3. Lista manual como fallback final

In [ ]:
trigger_dates = None

# ── Tentativa 1: SharePoint ───────────────────────────────────────────────
_ev_path = Path('../sharepoint.ev')
if _ev_path.exists():
    _env = {}
    for line in _ev_path.read_text().splitlines():
        if '=' in line:
            k, v = line.strip().split('=', 1)
            _env[k.strip()] = v.strip()
    try:
        from src.sharepoint_methods import SharePointClient
        _sp_url  = _env.get('SP_URL', 'https://kimberlyclark.sharepoint.com/Sites/H945')
        _sp = SharePointClient(_sp_url, _env['SP_USER'], _env['SP_PASS'])
        df_sp = _sp.query_large_list('Gatilhos_Selagem')
        _dates = df_sp['Title'].str.extract(r'(\d{4}-\d{2}-\d{2})', expand=False)
        trigger_dates = pd.to_datetime(_dates.dropna(), utc=True)
        trigger_dates = trigger_dates[trigger_dates >= pd.Timestamp(TRIGGER_DESDE, tz='UTC')]
        trigger_dates = trigger_dates.sort_values().drop_duplicates()
        print(f'Disparos via SharePoint   : {len(trigger_dates)}')
        pd.DataFrame({'disparo_date': trigger_dates.dt.date}).to_csv('trigger_dates.csv', index=False)
        print('  → trigger_dates.csv salvo')
    except Exception as e:
        print(f'[WARN] SharePoint: {e}')
        trigger_dates = None
else:
    print('sharepoint.ev não encontrado — pulando tentativa SP')

# ── Tentativa 2: anomaly_delay.csv ────────────────────────────────────────
if trigger_dates is None or len(trigger_dates) == 0:
    _anom_path = Path('anomaly_delay.csv')
    if _anom_path.exists():
        df_anom = pd.read_csv(_anom_path, parse_dates=['Timestamp'])
        df_anom['Timestamp'] = pd.to_datetime(df_anom['Timestamp'], utc=True)
        _mask = (
            df_anom['is_yellow_card'] &
            (df_anom['Timestamp'] >= pd.Timestamp(TRIGGER_DESDE, tz='UTC'))
        )
        trigger_dates = (
            df_anom.loc[_mask, 'Timestamp']
            .dt.normalize()
            .drop_duplicates()
            .sort_values()
        )
        print(f'Disparos via anomaly_delay.csv: {len(trigger_dates)}')
    else:
        print('⚠️  anomaly_delay.csv não encontrado')

# ── Fallback manual ───────────────────────────────────────────────────────
if trigger_dates is None or len(trigger_dates) == 0:
    trigger_dates = pd.to_datetime([
        # Insira datas manualmente se necessário, ex:
        # '2025-03-15', '2025-06-01',
    ], utc=True)
    print(f'Usando {len(trigger_dates)} datas manuais')

print(f'\nTotal de disparos: {len(trigger_dates)}')
for d in trigger_dates:
    print(f'  {d.date()}')

## 2. Buscar PI points

- Exclui sinais com `Capsule` no nome
- Mantém apenas sinais do tipo `*Signal*` (inclui `StoredSignal` e `CalculatedSignal`; exclui `Condition`, `Scalar`)
- Imprime as colunas e tipos únicos retornados pelo `spy.search` para diagnóstico

In [ ]:
print(f'Buscando sinais com padrão: {SEARCH_PATTERN} ...')
df_search = spy.search({'Name': SEARCH_PATTERN}, quiet=True)

# ── Diagnóstico: o que spy.search retornou ────────────────────────────────
print(f'\nColunas retornadas pelo spy.search:')
print(f'  {list(df_search.columns)}')
print(f'\nTotal de resultados (raw): {len(df_search)}')

if 'Type' in df_search.columns:
    print(f'\nTipos únicos (coluna Type):')
    for t, cnt in df_search['Type'].value_counts().items():
        print(f'  {t:<30} : {cnt}')

if 'Datasource Name' in df_search.columns:
    print(f'\nDatasources únicos:')
    for ds, cnt in df_search['Datasource Name'].value_counts().items():
        print(f'  {str(ds)[:50]:<50} : {cnt}')

# ── Filtrar Capsules ──────────────────────────────────────────────────────
if EXCLUIR_CAPSULES:
    _mask_cap = df_search['Name'].str.contains('capsule', case=False, na=False)
    n_cap = _mask_cap.sum()
    df_search = df_search[~_mask_cap].copy()
    print(f'\nApós filtro Capsule (-{n_cap}): {len(df_search)} sinais')

# ── Filtrar apenas tipos Signal ───────────────────────────────────────────
# spy.search retorna 'StoredSignal', 'CalculatedSignal' etc. — nunca apenas 'Signal'
if 'Type' in df_search.columns:
    _n_antes = len(df_search)
    _mask_signal = df_search['Type'].str.lower().str.contains('signal', na=False)
    df_search = df_search[_mask_signal].copy()
    print(f'Após filtro type=*Signal* (-{_n_antes - len(df_search)}): {len(df_search)} sinais')

# ── DataFrame final ───────────────────────────────────────────────────────
_keep_cols = [c for c in ['ID', 'Name', 'Type', 'Datasource Name', 'Value Unit Of Measure']
              if c in df_search.columns]
df_signals = df_search[_keep_cols].drop_duplicates(subset='ID').reset_index(drop=True)

print(f'\nTotal para análise: {len(df_signals)} sinais')
print(df_signals[['Name', 'Type']].head(10).to_string())

## 3. Inspecionar PI points individualmente

Função `inspect_pi_point()` — pull bruto de um sinal para confirmar se tem dados e medir a granularidade real.

Use para depurar sinais suspeitos antes de incluí-los no pull em lote.

In [ ]:
def pull_single(signal_id: str, signal_type: str, start: str, end: str,
                grid=None, header: str = 'Name') -> pd.DataFrame:
    """
    Padrão canônico validado: pull de um único sinal por vez.

    Puxa um ID isolado e retorna DataFrame com índice UTC.
    Retorna DataFrame vazio em caso de erro ou resultado vazio — nunca lança exceção.

    Args:
        signal_id   : UUID Seeq
        signal_type : Type real do spy.search ('StoredSignal', 'CalculatedSignal', etc.)
        start / end : strings .isoformat() com timezone
        grid        : '1h', '1d', None (bruto)
        header      : 'Name' ou 'ID'
    """
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            df = spy.pull(
                pd.DataFrame([{'ID': signal_id, 'Type': signal_type}]),
                start=start,
                end=end,
                grid=grid,
                header=header,
                quiet=True,
            )
        if df is not None and not df.empty:
            df.index = pd.to_datetime(df.index, utc=True)
            return df
    except Exception:
        pass
    return pd.DataFrame()


def inspect_pi_point(signal_id: str, signal_type: str = 'Signal',
                     days: int = 7, grid=None) -> pd.DataFrame:
    """
    Puxa dados brutos de um PI point para inspecionar.

    Uso:
        df = inspect_pi_point('D879CAD1-0BD1-4BDA-B215-6FF0E203A8D4', days=14)
        print(df.head())
        df.plot(figsize=(15, 5))
    """
    _tz   = spy.session.get_user_timezone()
    end   = pd.Timestamp.now(tz=_tz)
    start = end - pd.DateOffset(days=days)
    df = pull_single(signal_id, signal_type, start.isoformat(), end.isoformat(), grid=grid)
    if df.empty:
        print(f'  [VAZIO] {signal_id}: sem dados nos últimos {days} dias')
    return df


def resume_pi_point(signal_id: str, signal_type: str = 'Signal',
                    signal_name: str = '', days: int = 30) -> dict:
    """Resumo rápido: tem dados? Granularidade? Faixa de valores?"""
    df = inspect_pi_point(signal_id, signal_type, days=days, grid=None)
    if df.empty:
        return {'id': signal_id, 'name': signal_name, 'tem_dados': False,
                'n_leituras': 0, 'gran_mediana_s': None, 'min': None, 'max': None}

    col  = df.columns[0]
    vals = df[col].dropna()
    intervalos = df.index.to_series().diff().dt.total_seconds().dropna()
    gran = round(intervalos.median(), 1) if len(intervalos) > 0 else None

    return {
        'id'            : signal_id,
        'name'          : signal_name or col,
        'tem_dados'     : len(vals) > 0,
        'n_leituras'    : len(df),
        'n_nao_nulos'   : len(vals),
        'gran_mediana_s': gran,
        'min'           : round(float(vals.min()), 4) if len(vals) > 0 else None,
        'max'           : round(float(vals.max()), 4) if len(vals) > 0 else None,
        'p50'           : round(float(vals.median()), 4) if len(vals) > 0 else None,
    }


print('Funções carregadas: pull_single(), inspect_pi_point(), resume_pi_point()')
print()
print('Uso rápido:')
print('  df = inspect_pi_point("D879CAD1-...", "StoredSignal", days=7)')
print('  r  = resume_pi_point("D879CAD1-...", "StoredSignal", days=30)')

In [ ]:
# Diagnóstico: amostrar os primeiros N sinais encontrados (últimos 30 dias, dados brutos)
N_AMOSTRA_DIAG = 10

print(f'Inspecionando {N_AMOSTRA_DIAG} sinais (últimos 30 dias, grid=None)...\n')

resumos = []
for _, row in df_signals.head(N_AMOSTRA_DIAG).iterrows():
    sig_type = row.get('Type', 'Signal')
    r = resume_pi_point(row['ID'], sig_type, row['Name'], days=30)
    resumos.append(r)
    icon     = '✅' if r['tem_dados'] else '❌'
    gran_str = f"{r['gran_mediana_s']}s" if r['gran_mediana_s'] else 'N/A'
    print(f"  {icon} {r['name'][:55]:<55} | {r['n_leituras']:>5} leit | gran={gran_str:<8} | [{r['min']}, {r['max']}]")

df_diag = pd.DataFrame(resumos)
n_com = df_diag['tem_dados'].sum()
print(f'\nResultado: {n_com}/{N_AMOSTRA_DIAG} sinais com dados nos últimos 30 dias')

## 4. Check de disponibilidade em lote — pilot pull

Pull com `grid=1d` no período baseline, em batches de `BATCH_SIZE` sinais.
Descarta sinais com menos de `MIN_COBERTURA` de valores não-nulos.

> Se o piloto mostrar 0% de cobertura em sinais que você sabe que têm dados,
> reduza `BATCH_SIZE` ou use `resume_pi_point()` na célula anterior para inspecionar.

In [ ]:
print(f'Pilot pull ({DATA_INICIO.date()} → {DATA_BASELINE_END.date()}, grid={PILOT_GRID})')
print(f'  {len(df_signals)} sinais — pull individual por sinal...\n')

ids_ativos = []

for i, (_, row) in enumerate(df_signals.iterrows()):
    sig_type = row.get('Type', 'Signal')
    df_p = pull_single(
        row['ID'], sig_type,
        DATA_INICIO.isoformat(),
        DATA_BASELINE_END.isoformat(),
        grid=PILOT_GRID,
    )
    if not df_p.empty:
        cob = df_p.notna().mean().mean()
        if cob >= MIN_COBERTURA:
            ids_ativos.append(row['ID'])

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(df_signals)} verificados | {len(ids_ativos)} ativos')

df_signals_ativos = (
    df_signals[df_signals['ID'].isin(ids_ativos)]
    .reset_index(drop=True)
)

print(f'\nAtivos (cob >= {MIN_COBERTURA:.0%}): {len(df_signals_ativos)}')
print(f'Descartados (vazios/esparsos)     : {len(df_signals) - len(df_signals_ativos)}')

## 5. Pull contínuo — ano vigente em grid=1h

Um pull por sinal, período completo `[1 jan → agora]`. Baseline e janelas de
anomalia são fatiadas localmente na célula seguinte — zero chamadas extras ao Seeq.

**Custo:** `N_sinais_ativos` chamadas. Após esta célula, todas as análises são pandas puro.

In [ ]:
print(f'Pull contínuo: {DATA_INICIO.date()} → {DATA_FIM.date()}, grid={GRID}')
print(f'  {len(df_signals_ativos)} sinais ativos — pull individual por sinal...\n')

continuo_chunks = {}  # ID → DataFrame

for i, (_, row) in enumerate(df_signals_ativos.iterrows()):
    sig_type = row.get('Type', 'Signal')
    df_c = pull_single(
        row['ID'], sig_type,
        DATA_INICIO.isoformat(),
        DATA_FIM.isoformat(),
        grid=GRID,
        header='ID',
    )
    if not df_c.empty:
        continuo_chunks[row['ID']] = df_c

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(df_signals_ativos)} | {len(continuo_chunks)} com dados')

if not continuo_chunks:
    raise RuntimeError('Pull contínuo falhou em todos os sinais — verifique conexão Seeq.')

df_continuo = pd.concat(list(continuo_chunks.values()), axis=1)
df_continuo.index = pd.to_datetime(df_continuo.index, utc=True)

_mb = df_continuo.memory_usage(deep=True).sum() / 1e6
print(f'\ndf_continuo: {len(df_continuo):,} leituras × {len(df_continuo.columns)} sinais ({_mb:.0f} MB)')

## 6. Fatiar baseline e janelas — pandas puro, zero chamadas ao Seeq

Baseline e janelas de anomalia são extraídas de `df_continuo` via indexação temporal.
Nenhuma chamada adicional ao Seeq — toda a análise roda localmente.

In [ ]:
# ── Fatiar baseline ───────────────────────────────────────────────────────
_bs = DATA_BASELINE_START.tz_convert('UTC')
_be = DATA_BASELINE_END.tz_convert('UTC').replace(hour=23, minute=59)

df_baseline_raw = df_continuo.loc[_bs:_be]

baseline_stats = pd.DataFrame({
    'mean':  df_baseline_raw.mean(),
    'std':   df_baseline_raw.std().clip(lower=1e-9),
    'p10':   df_baseline_raw.quantile(0.10),
    'p90':   df_baseline_raw.quantile(0.90),
    'n_obs': df_baseline_raw.notna().sum(),
})

print(f'Baseline fatiado: {DATA_BASELINE_START.date()} → {DATA_BASELINE_END.date()}')
print(f'  {len(df_baseline_raw):,} leituras | {(baseline_stats["n_obs"] >= 5).sum()} sinais com >= 5 obs')

# ── Fatiar janelas de anomalia ─────────────────────────────────────────────
print(f'\nFatiando {len(trigger_dates)} janelas de anomalia...')

window_stats = {}

for tdate in trigger_dates:
    t_start = (tdate - pd.Timedelta(days=JANELA_ANTES_D)).tz_convert('UTC')
    t_end   = (tdate + pd.Timedelta(days=JANELA_DEPOIS_D, hours=23, minutes=59)).tz_convert('UTC')
    label   = str(tdate.date())

    df_win = df_continuo.loc[t_start:t_end]

    if df_win.empty:
        print(f'  {label}: sem dados no df_continuo (disparo fora do ano vigente?)')
        continue

    window_stats[label] = {
        'mean':  df_win.mean(),
        'n_obs': df_win.notna().sum(),
    }
    n_com = (df_win.notna().sum() > 0).sum()
    print(f'  {label}: {len(df_win):,} leituras | {n_com} sinais com dados  ✓')

print(f'\nJanelas fatiadas: {len(window_stats)}/{len(trigger_dates)}')

## 7. Comparação — z-score por sinal × janela

```
z = (media_janela − baseline_mean) / baseline_std
```

Sinal é classificado como anômalo se `|z| >= ZSCORE_LIMIAR` em pelo menos uma janela.

In [ ]:
_id_to_name = dict(zip(df_signals_ativos['ID'], df_signals_ativos['Name']))

records = []

for signal_id in baseline_stats.index:
    b_mean = baseline_stats.loc[signal_id, 'mean']
    b_std  = baseline_stats.loc[signal_id, 'std']
    b_n    = baseline_stats.loc[signal_id, 'n_obs']

    if pd.isna(b_mean) or b_n < 5:
        continue

    rec = {
        'Signal_ID':     signal_id,
        'Signal_Name':   _id_to_name.get(signal_id, signal_id),
        'Baseline_Mean': round(float(b_mean), 4),
        'Baseline_Std':  round(float(b_std),  4),
        'Baseline_N':    int(b_n),
    }

    zscores_abs = []
    for label, wstats in window_stats.items():
        w_mean = wstats['mean'].get(signal_id, float('nan'))
        w_n    = int(wstats['n_obs'].get(signal_id, 0))
        if pd.isna(w_mean) or w_n == 0:
            z = float('nan')
        else:
            z = (float(w_mean) - float(b_mean)) / float(b_std)
        rec[f'ZScore_{label}'] = round(z, 3) if not np.isnan(z) else None
        if not np.isnan(z):
            zscores_abs.append(abs(z))

    rec['Max_AbsZScore']      = round(max(zscores_abs), 3) if zscores_abs else float('nan')
    rec['N_Windows_Anomalos'] = sum(1 for z in zscores_abs if z >= ZSCORE_LIMIAR)
    records.append(rec)

df_result = (
    pd.DataFrame(records)
    .sort_values('Max_AbsZScore', ascending=False)
    .reset_index(drop=True)
)

n_anomalos = (df_result['N_Windows_Anomalos'] > 0).sum()
print(f'Sinais analisados                      : {len(df_result)}')
print(f'Anômalos (|z|>={ZSCORE_LIMIAR}, >=1 janela): {n_anomalos}')

cols_show = ['Signal_Name', 'Baseline_Mean', 'Baseline_Std', 'Max_AbsZScore', 'N_Windows_Anomalos']
print(f'\nTop 20 por Max_AbsZScore:')
print(df_result.head(20)[cols_show].to_string(index=False))

## 8. Detalhe — sinais anômalos em múltiplas janelas

In [ ]:
df_multi = df_result[df_result['N_Windows_Anomalos'] > 1].copy()
print(f'Sinais anômalos em > 1 janela: {len(df_multi)}')

if not df_multi.empty:
    z_cols = [c for c in df_multi.columns if c.startswith('ZScore_')]
    print(df_multi[['Signal_Name', 'N_Windows_Anomalos', 'Max_AbsZScore'] + z_cols].to_string(index=False))

# Direção: sinais consistentemente acima vs. abaixo do normal
if window_stats:
    _z_cols = [c for c in df_result.columns if c.startswith('ZScore_')]
    if _z_cols:
        _df_z = df_result.set_index('Signal_Name')[_z_cols].apply(pd.to_numeric, errors='coerce')
        _mean_z = _df_z.mean(axis=1).dropna()

        print(f'\nTop 5 consistentemente ACIMA do normal durante disparos:')
        print(_mean_z.nlargest(5).to_string())
        print(f'\nTop 5 consistentemente ABAIXO do normal durante disparos:')
        print(_mean_z.nsmallest(5).to_string())

## 9. Exportar anomaly_scan.csv

In [ ]:
df_result.to_csv(OUTPUT_CSV, index=False)

print(f'Exportado: {OUTPUT_CSV}')
print(f'  Sinais            : {len(df_result)}')
print(f'  Anômalos          : {(df_result["N_Windows_Anomalos"] > 0).sum()}')
print(f'  Janelas analisadas: {len(window_stats)}')
print(f'  Colunas           : {list(df_result.columns)}')

print(f'\nAnômalos por janela (|z| >= {ZSCORE_LIMIAR}):')
for label in window_stats:
    col = f'ZScore_{label}'
    if col in df_result.columns:
        n = df_result[col].apply(lambda x: abs(x) >= ZSCORE_LIMIAR if x is not None else False).sum()
        print(f'  {label}: {n} sinais anômalos')